# Hospitality Management Analytics - Bronze to Silver Transformation

This notebook transforms raw bronze layer data into cleaned and conformed silver layer tables.

**Silver Tables to Create:**
1. `dim_guests` - Guest dimension with SCD Type 2 for loyalty tier changes
2. `fact_stays_unified` - Unified reservation and POS transaction fact table
3. `fact_room_availability_daily` - Daily room availability calendar

**Author:** Data Engineering Team

**Date:** January 2025

## Step 1: Environment Setup

In [0]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, trim, upper, lower, regexp_replace, to_date,
    coalesce, lit, current_timestamp, row_number, dense_rank, lag, lead,
    datediff, explode, sequence, date_add, sum as spark_sum, count, avg,
    max as spark_max, min as spark_min, expr, concat_ws, md5, unix_timestamp,
    dayofweek, year, month, dayofmonth
)
# Note: We use expr() with try_to_timestamp SQL function for safe timestamp parsing
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType, TimestampType

# Define paths and names
CATALOG_NAME = 'hospitality_project'
BRONZE_SCHEMA = 'bronze_schema'
SILVER_SCHEMA = 'silver_schema'
VOLUME_NAME = 'raw'
VOLUME_PATH = f'/Volumes/{CATALOG_NAME}/{BRONZE_SCHEMA}/{VOLUME_NAME}'
CHECKPOINT_PATH = f'/Volumes/{CATALOG_NAME}/{BRONZE_SCHEMA}/checkpoints'

print(f'Volume Path: {VOLUME_PATH}')
print(f'Checkpoint Path: {CHECKPOINT_PATH}')

## Step 2: Create Silver Schema

In [0]:
%sql
-- Create silver schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS hospitality_project.silver_schema;

## Step 3: Read Bronze Data

We'll read JSON files from the bronze layer volumes.

In [0]:
def read_bronze_data(folder_name, schema_location):
    """
    Read bronze data from JSON files.
    
    Parameters:
    - folder_name: Name of the folder in the volume (e.g., 'guests')
    - schema_location: Path to store inferred schema
    
    Returns:
    - DataFrame with bronze data
    """
    path = f'{VOLUME_PATH}/{folder_name}'
    
    df = (spark.read
        .format("json")
        .option("inferSchema", "true")
        .option("multiLine", "false")
        .load(path))
    
    print(f'Read {df.count()} records from {folder_name}')
    return df

# Read all bronze tables
print('\n=== Reading Bronze Data ===')
bronze_guests = read_bronze_data('guests', f'{CHECKPOINT_PATH}/guests_schema')
bronze_inventory = read_bronze_data('hotel_inventory', f'{CHECKPOINT_PATH}/inventory_schema')
bronze_reservations = read_bronze_data('reservations', f'{CHECKPOINT_PATH}/reservations_schema')
bronze_pos = read_bronze_data('pos_transactions', f'{CHECKPOINT_PATH}/pos_schema')
bronze_housekeeping = read_bronze_data('housekeeping_logs', f'{CHECKPOINT_PATH}/housekeeping_schema')

In [0]:
# Display sample data and schema
print('\n=== Bronze Guests Schema ===')
bronze_guests.printSchema()
bronze_guests.show(5, truncate=False)

## Step 4: Transform to Silver - dim_guests (SCD Type 2)

This dimension tracks guest information with SCD Type 2 for loyalty tier changes.

In [0]:
print('\n=== Transforming dim_guests (SCD Type 2) ===')

# Step 1: Clean and standardize guest data
guests_cleaned = (
    bronze_guests
    # Filter out records with missing critical fields
    .filter(col('guest_id').isNotNull())
    
    # Clean and standardize name (trim whitespace, proper case)
    .withColumn('name', 
        when(col('name').isNotNull(), 
             regexp_replace(trim(col('name')), '\\s+', ' '))
        .otherwise(lit('Unknown')))
    
    # Clean and standardize email (lowercase, trim)
    .withColumn('email',
        when(col('email').isNotNull() & (col('email') != 'invalid-email'),
             lower(trim(col('email'))))
        .otherwise(None))
    
    # Standardize loyalty_tier (proper case)
    .withColumn('loyalty_tier',
        when(col('loyalty_tier').isNotNull(),
             when(upper(col('loyalty_tier')) == 'BRONZE', 'Bronze')
             .when(upper(col('loyalty_tier')) == 'SILVER', 'Silver')
             .when(upper(col('loyalty_tier')) == 'GOLD', 'Gold')
             .when(upper(col('loyalty_tier')) == 'PLATINUM', 'Platinum')
             .otherwise('Bronze'))
        .otherwise('Bronze'))
    
    # Clean country (trim, proper case)
    .withColumn('country',
        when(col('country').isNotNull(), upper(trim(col('country'))))
        .otherwise(lit('UNKNOWN')))
    
    # Parse registration_date (handle multiple formats) - FIXED
    .withColumn('registration_date',
        expr("""
            COALESCE(
                try_to_date(registration_date, 'yyyy-MM-dd'),
                try_to_date(registration_date, 'dd/MM/yyyy'),
                try_to_date(registration_date, 'MM/dd/yyyy')
            )
        """))
    
    # Parse updated_at timestamp (handle multiple formats) - FIXED ESCAPING
    .withColumn('updated_at',
        expr("""
            COALESCE(
                try_to_timestamp(updated_at, "yyyy-MM-dd'T'HH:mm:ss'Z'"),
                try_to_timestamp(updated_at, 'dd/MM/yyyy HH:mm'),
                try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss')
            )
        """))
)

print(f'Cleaned guests: {guests_cleaned.count()} records')

In [0]:
# Step 2: Deduplicate guests (identify same guest with email variations)
# Create a deduplication key based on email (normalized)
guests_dedup_prep = (
    guests_cleaned
    # Extract email base (before +)
    .withColumn('email_base',
        when(col('email').isNotNull(),
             regexp_replace(col('email'), '\\+.*@', '@'))
        .otherwise(None))
    
    # Create deduplication key: email_base or guest_id if email is null
    .withColumn('dedup_key',
        coalesce(
            col('email_base'),
            concat_ws('_', lit('guest'), col('guest_id'))
        ))
)

# Deduplicate: Keep the record with the latest updated_at for each dedup_key
# FIXED: Handle NULL updated_at values properly
window_dedup = Window.partitionBy('dedup_key').orderBy(
    col('updated_at').desc_nulls_last(), 
    col('guest_id').desc()
)

guests_deduped = (
    guests_dedup_prep
    .withColumn('row_num', row_number().over(window_dedup))
    .filter(col('row_num') == 1)
    .drop('row_num', 'email_base', 'dedup_key')
    # Filter out records where updated_at is NULL (critical for SCD Type 2)
    .filter(col('updated_at').isNotNull())
)

print(f'After deduplication: {guests_deduped.count()} unique guests')

In [0]:
# Step 3: Implement SCD Type 2 for loyalty_tier changes
# Create version history based on loyalty_tier and updated_at

# FIXED: Add explicit NULL handling in window ordering
window_tier = Window.partitionBy('guest_id').orderBy(
    col('updated_at').asc_nulls_last()
)

guests_scd = (
    guests_deduped
    # Identify tier changes by comparing with previous tier
    .withColumn('prev_tier', lag('loyalty_tier').over(window_tier))
    .withColumn('tier_changed',
        when((col('prev_tier').isNotNull()) & (col('prev_tier') != col('loyalty_tier')), lit(1))
        .otherwise(lit(0)))
    
    # Create version number for each tier change
    .withColumn('version', 
        spark_sum('tier_changed').over(
            window_tier.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        ) + 1)
    
    # Set valid_from as updated_at (already properly formatted)
    .withColumn('valid_from', col('updated_at'))
    
    # Set valid_to as next updated_at or null for current record
    .withColumn('valid_to', lead('updated_at').over(window_tier))
    
    # Mark current record
    .withColumn('is_current', when(col('valid_to').isNull(), lit(True)).otherwise(lit(False)))
    
    # Create surrogate key: guest_id + version
    .withColumn('guest_sk', concat_ws('_', col('guest_id').cast('string'), col('version').cast('string')))
    
    # Drop temporary columns
    .drop('prev_tier', 'tier_changed')
    
    # Select final columns - ensure all are properly typed
    .select(
        col('guest_sk'),
        col('guest_id'),
        col('name'),
        col('email'),
        col('loyalty_tier'),
        col('country'),
        col('registration_date').cast('date'),  # Explicit cast
        col('valid_from').cast('timestamp'),    # Explicit cast
        col('valid_to').cast('timestamp'),      # Explicit cast
        col('is_current'),
        col('version')
    )
)

print(f'SCD Type 2 dim_guests: {guests_scd.count()} records (includes historical versions)')

# Show sample - FIXED: Limit and specific columns to avoid display issues
print('\nSample dim_guests (SCD Type 2):')
guests_scd.select(
    'guest_sk', 'guest_id', 'name', 'loyalty_tier', 
    'valid_from', 'valid_to', 'is_current', 'version'
).orderBy('guest_id', 'version').show(10, truncate=False)

# Show tier change examples
print('\nGuests with tier changes (multiple versions):')
guests_with_changes = (
    guests_scd
    .groupBy('guest_id')
    .agg(count('*').alias('version_count'))
    .filter(col('version_count') > 1)
    .select('guest_id')
)
guests_scd.join(guests_with_changes, 'guest_id').orderBy('guest_id', 'version').show(10, truncate=False)

In [0]:
# Write dim_guests to Delta table
print('\n=== Writing dim_guests to Delta ===')

(guests_scd
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(f'{CATALOG_NAME}.{SILVER_SCHEMA}.dim_guests'))

print('✅ dim_guests table created successfully!')

## Step 5: Transform to Silver - fact_stays_unified

This fact table unifies reservations with POS transactions to create total folio amounts.

In [0]:
print('\n=== Transforming fact_stays_unified ===')

# Step 1: Clean and standardize reservations
reservations_cleaned = (
    bronze_reservations
    # Filter out records with missing critical fields
    .filter(col('res_id').isNotNull())
    
    # Standardize room_type (proper case)
    .withColumn('room_type',
        when(col('room_type').isNotNull(),
             when(upper(col('room_type')) == 'STANDARD', 'Standard')
             .when(upper(col('room_type')) == 'DELUXE', 'Deluxe')
             .when(upper(col('room_type')) == 'SUITE', 'Suite')
             .when(upper(col('room_type')) == 'PRESIDENTIAL', 'Presidential')
             .otherwise('Standard'))
        .otherwise('Standard'))
    
    # Standardize booking_channel (proper case)
    .withColumn('booking_channel',
        when(col('booking_channel').isNotNull(),
             when(upper(col('booking_channel')) == 'WEBSITE', 'Website')
             .when(upper(col('booking_channel')) == 'PHONE', 'Phone')
             .when(upper(col('booking_channel')).contains('WALK'), 'Walk-in')
             .when(upper(col('booking_channel')) == 'OTA', 'OTA')
             .otherwise('Unknown'))
        .otherwise('Unknown'))
    
    # Parse check_in_date (handle multiple formats) - FIXED
    .withColumn('check_in_date',
        coalesce(
            expr("try_to_date(check_in_date, 'yyyy-MM-dd')"),
            expr("try_to_date(check_in_date, 'dd/MM/yyyy')"),
            expr("try_to_date(check_in_date, 'MM/dd/yyyy')")
        ))
    
    # Parse check_out_date (handle multiple formats) - FIXED
    .withColumn('check_out_date',
        coalesce(
            expr("try_to_date(check_out_date, 'yyyy-MM-dd')"),
            expr("try_to_date(check_out_date, 'dd/MM/yyyy')"),
            expr("try_to_date(check_out_date, 'MM-dd-yyyy')"),
            expr("try_to_date(check_out_date, 'MM/dd/yyyy')")
        ))
    
    # Parse created_at timestamp - FIXED
    .withColumn('created_at',
        coalesce(
            expr("try_to_timestamp(created_at, \"yyyy-MM-dd'T'HH:mm:ss'Z'\")"),
            expr("try_to_timestamp(created_at, 'dd/MM/yyyy HH:mm')"),
            expr("try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss')")
        ))
    
    # Clean total_price (handle negative and null values)
    .withColumn('total_price',
        when((col('total_price').isNotNull()) & (col('total_price') > 0) & (col('total_price') < 10000),
             col('total_price'))
        .otherwise(None))
    
    # Filter out records with invalid dates
    .filter(
        (col('check_in_date').isNotNull()) &
        (col('check_out_date').isNotNull()) &
        (col('check_out_date') > col('check_in_date'))  # check_out must be after check_in
    )
)

print(f'Cleaned reservations: {reservations_cleaned.count()} records')

In [0]:
# Step 2: Remove duplicate reservations (keep latest by created_at)
window_res_dedup = Window.partitionBy('res_id').orderBy(col('created_at').desc_nulls_last())

reservations_deduped = (
    reservations_cleaned
    .withColumn('row_num', row_number().over(window_res_dedup))
    .filter(col('row_num') == 1)
    .drop('row_num')
)

print(f'After deduplication: {reservations_deduped.count()} unique reservations')

In [0]:
# Step 3: Clean and standardize POS transactions
pos_cleaned = (
    bronze_pos
    # Filter out records with missing critical fields
    .filter(col('txn_id').isNotNull())
    
    # Standardize category (proper case)
    .withColumn('category',
        when(col('category').isNotNull(),
             when(upper(col('category')) == 'FOOD', 'Food')
             .when(upper(col('category')) == 'DRINK', 'Drink')
             .when(upper(col('category')) == 'SERVICE', 'Service')
             .when(upper(col('category')) == 'SPA', 'Spa')
             .otherwise('Other'))
        .otherwise('Other'))
    
    # Parse timestamp (handle multiple formats) - FIXED
    .withColumn('timestamp',
        coalesce(
            expr("try_to_timestamp(timestamp, \"yyyy-MM-dd'T'HH:mm:ss'Z'\")"),
            expr("try_to_timestamp(timestamp, 'dd/MM/yyyy HH:mm')"),
            expr("try_to_timestamp(timestamp, 'yyyy-MM-dd HH:mm:ss')")
        ))
    
    # Clean amount (handle negative and extreme values)
    .withColumn('amount',
        when((col('amount').isNotNull()) & (col('amount') > 0) & (col('amount') < 5000),
             col('amount'))
        .otherwise(None))
    
    # Filter valid transactions
    .filter(
        (col('amount').isNotNull()) &
        (col('timestamp').isNotNull())
    )
)

print(f'Cleaned POS transactions: {pos_cleaned.count()} records')

In [0]:
# Step 4: Aggregate POS transactions by res_id (exclude walk-ins with NULL res_id)
pos_aggregated = (
    pos_cleaned
    .filter(col('res_id').isNotNull())  # Only reservations, exclude walk-ins
    .groupBy('res_id')
    .agg(
        spark_sum('amount').alias('total_pos_amount'),
        count('*').alias('pos_item_count')
    )
)

print(f'Aggregated POS by reservation: {pos_aggregated.count()} reservations with POS')

In [0]:
# Step 5: Join reservations with POS aggregations
fact_stays = (
    reservations_deduped
    .join(pos_aggregated, on='res_id', how='left')
    
    # Fill nulls for reservations without POS transactions
    .withColumn('total_pos_amount', coalesce(col('total_pos_amount'), lit(0.0)))
    .withColumn('pos_item_count', coalesce(col('pos_item_count'), lit(0)))
    
    # Calculate total folio amount
    .withColumn('total_folio_amount',
        when(col('total_price').isNotNull(),
             col('total_price') + col('total_pos_amount'))
        .otherwise(col('total_pos_amount')))
    
    # Calculate stay length (nights)
    .withColumn('stay_length_nights', datediff(col('check_out_date'), col('check_in_date')))
    
    # Select final columns
    .select(
        'res_id',
        'guest_id',
        'hotel_id',
        'room_type',
        'check_in_date',
        'check_out_date',
        'stay_length_nights',
        'total_price',
        'booking_channel',
        'total_pos_amount',
        'total_folio_amount',
        'pos_item_count',
        'created_at'
    )
)

print(f'Unified stays fact table: {fact_stays.count()} records')

In [0]:
# Step 6: Filter orphaned records (guest_id or hotel_id not in dimensions)
# Get valid guest_ids from dim_guests
valid_guest_ids = spark.table(f'{CATALOG_NAME}.{SILVER_SCHEMA}.dim_guests').select('guest_id').distinct()

# Get valid hotel_ids from bronze inventory
valid_hotel_ids = bronze_inventory.select('hotel_id').distinct()

fact_stays_validated = (
    fact_stays
    .join(valid_guest_ids, on='guest_id', how='inner')  # Remove orphaned guests
    .join(valid_hotel_ids, on='hotel_id', how='inner')  # Remove orphaned hotels
)

print(f'After removing orphaned records: {fact_stays_validated.count()} records')

# Show sample
print('\nSample fact_stays_unified:')
fact_stays_validated.orderBy('res_id').show(10, truncate=False)

In [0]:
# Write fact_stays_unified to Delta table
print('\n=== Writing fact_stays_unified to Delta ===')

(fact_stays_validated
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('check_in_date')  # Partition by check_in_date for performance
    .saveAsTable(f'{CATALOG_NAME}.{SILVER_SCHEMA}.fact_stays_unified'))

print('✅ fact_stays_unified table created successfully!')

## Step 6: Transform to Silver - fact_room_availability_daily

This fact table explodes reservations into daily rows (date-split exercise).

In [0]:
print('\n=== Transforming fact_room_availability_daily ===')

# Step 1: Clean hotel inventory
inventory_cleaned = (
    bronze_inventory
    # Filter valid records
    .filter(
        (col('hotel_id').isNotNull()) &
        (col('room_number').isNotNull()) &
        (col('room_number') != '9999')  # Remove invalid room numbers
    )
    
    # Standardize room_type
    .withColumn('room_type',
        when(col('room_type').isNotNull(),
             when(upper(col('room_type')) == 'STANDARD', 'Standard')
             .when(upper(col('room_type')) == 'DELUXE', 'Deluxe')
             .when(upper(col('room_type')) == 'SUITE', 'Suite')
             .when(upper(col('room_type')) == 'PRESIDENTIAL', 'Presidential')
             .otherwise('Standard'))
        .otherwise('Standard'))
    
    .select('hotel_id', 'room_number', 'room_type', 'floor', 'max_occupancy')
)

print(f'Cleaned inventory: {inventory_cleaned.count()} rooms')

In [0]:
# Step 2: Get reservations with valid room assignments
reservations_with_rooms = (
    reservations_deduped
    .filter(col('room_number').isNotNull())
    .select(
        'res_id',
        'guest_id',
        'hotel_id',
        'room_number',
        'room_type',
        'check_in_date',
        'check_out_date'
    )
)

print(f'Reservations with room assignments: {reservations_with_rooms.count()} records')

In [0]:
# Step 3: Date explosion - Create one row per night of stay
# Use explode with sequence to generate dates between check_in and check_out
room_occupancy_daily = (
    reservations_with_rooms
    # Generate array of dates from check_in to check_out (exclusive)
    .withColumn('date_array',
        expr('sequence(check_in_date, date_sub(check_out_date, 1), interval 1 day)'))
    
    # Explode the date array to create one row per date
    .withColumn('date', explode(col('date_array')))
    
    # Mark as occupied
    .withColumn('is_available', lit(False))
    
    # Select final columns
    .select(
        'date',
        'hotel_id',
        'room_number',
        'room_type',
        'is_available',
        'res_id',
        'guest_id',
        'check_in_date',
        'check_out_date'
    )
)

print(f'Room occupancy (exploded daily): {room_occupancy_daily.count()} date-room records')

In [0]:
# Step 4: Detect overbookings (same hotel_id + room_number + date with multiple res_id)
window_overbooking = Window.partitionBy('hotel_id', 'room_number', 'date')

room_occupancy_flagged = (
    room_occupancy_daily
    .withColumn('booking_count', count('res_id').over(window_overbooking))
    .withColumn('is_overbooked', when(col('booking_count') > 1, lit(True)).otherwise(lit(False)))
)

overbooking_count = room_occupancy_flagged.filter(col('is_overbooked')).select('hotel_id', 'room_number', 'date').distinct().count()
print(f'⚠️  Detected {overbooking_count} overbooking scenarios (same room + date)')

In [0]:
# Step 5: For overbookings, keep only the first reservation (by res_id)
window_overbooking_resolve = Window.partitionBy('hotel_id', 'room_number', 'date').orderBy('res_id')

fact_room_availability = (
    room_occupancy_flagged
    .withColumn('row_num', row_number().over(window_overbooking_resolve))
    .filter(col('row_num') == 1)  # Keep first reservation for each room+date
    .drop('row_num', 'booking_count')
    
    # Select final columns
    .select(
        'date',
        'hotel_id',
        'room_number',
        'room_type',
        'is_available',
        'res_id',
        'guest_id',
        'check_in_date',
        'check_out_date',
        'is_overbooked'
    )
)

print(f'Final room availability (resolved overbookings): {fact_room_availability.count()} records')

# Show sample
print('\nSample fact_room_availability_daily:')
fact_room_availability.orderBy('hotel_id', 'room_number', 'date').show(10, truncate=False)

# Show overbooking examples
print('\nOverbooking examples:')
fact_room_availability.filter(col('is_overbooked')).show(5, truncate=False)

In [0]:
# Write fact_room_availability_daily to Delta table
print('\n=== Writing fact_room_availability_daily to Delta ===')

(fact_room_availability
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('date')  # Partition by date for query performance
    .saveAsTable(f'{CATALOG_NAME}.{SILVER_SCHEMA}.fact_room_availability_daily'))

print('✅ fact_room_availability_daily table created successfully!')

## Step 7: Data Quality Checks

In [0]:
print('\n' + '='*80)
print('DATA QUALITY REPORT - SILVER LAYER')
print('='*80)

# Check dim_guests
print('\n📊 dim_guests:')
dim_guests_df = spark.table(f'{CATALOG_NAME}.{SILVER_SCHEMA}.dim_guests')
print(f'  Total records: {dim_guests_df.count():,}')
print(f'  Unique guests: {dim_guests_df.select("guest_id").distinct().count():,}')
print(f'  Current records (is_current=True): {dim_guests_df.filter(col("is_current")).count():,}')
print(f'  Historical records: {dim_guests_df.filter(~col("is_current")).count():,}')

# Loyalty tier distribution
print('\n  Loyalty tier distribution (current records):')
dim_guests_df.filter(col('is_current')).groupBy('loyalty_tier').count().orderBy('loyalty_tier').show()

# Check fact_stays_unified
print('\n📊 fact_stays_unified:')
fact_stays_df = spark.table(f'{CATALOG_NAME}.{SILVER_SCHEMA}.fact_stays_unified')
print(f'  Total reservations: {fact_stays_df.count():,}')
print(f'  Reservations with POS: {fact_stays_df.filter(col("pos_item_count") > 0).count():,}')
print(f'  Reservations without POS: {fact_stays_df.filter(col("pos_item_count") == 0).count():,}')

# Calculate statistics
stats = fact_stays_df.select(
    avg('total_price').alias('avg_room_price'),
    avg('total_pos_amount').alias('avg_pos_amount'),
    avg('total_folio_amount').alias('avg_folio_amount'),
    avg('stay_length_nights').alias('avg_stay_length')
).first()

print(f'  Average room price: ${stats["avg_room_price"]:.2f}')
print(f'  Average POS amount: ${stats["avg_pos_amount"]:.2f}')
print(f'  Average total folio: ${stats["avg_folio_amount"]:.2f}')
print(f'  Average stay length: {stats["avg_stay_length"]:.1f} nights')

# Booking channel distribution
print('\n  Booking channel distribution:')
fact_stays_df.groupBy('booking_channel').count().orderBy(col('count').desc()).show()

# Check fact_room_availability_daily
print('\n📊 fact_room_availability_daily:')
fact_avail_df = spark.table(f'{CATALOG_NAME}.{SILVER_SCHEMA}.fact_room_availability_daily')
print(f'  Total date-room records: {fact_avail_df.count():,}')
print(f'  Overbookings detected: {fact_avail_df.filter(col("is_overbooked")).select("hotel_id", "room_number", "date").distinct().count():,}')

# Date range
date_range = fact_avail_df.select(
    spark_min('date').alias('min_date'),
    spark_max('date').alias('max_date')
).first()
print(f'  Date range: {date_range["min_date"]} to {date_range["max_date"]}')

# Room type distribution
print('\n  Room type distribution:')
fact_avail_df.groupBy('room_type').count().orderBy('room_type').show()

print('\n' + '='*80)
print('✅ Silver Layer Data Quality Checks Complete!')
print('='*80)

## Step 8: Late Checkout Analysis (Bonus)

In [0]:
print('\n=== Late Checkout POS Analysis ===')

# Identify POS transactions that occurred after checkout
late_checkout_analysis = (
    pos_cleaned
    .filter(col('res_id').isNotNull())
    .join(
        fact_stays_validated.select('res_id', 'check_out_date'),
        on='res_id',
        how='inner'
    )
    # Compare POS timestamp with checkout date
    .withColumn('pos_date', to_date(col('timestamp')))
    .withColumn('is_late_checkout',
        when(col('pos_date') >= col('check_out_date'), lit(True))
        .otherwise(lit(False)))
)

late_checkout_count = late_checkout_analysis.filter(col('is_late_checkout')).count()
total_pos_with_res = late_checkout_analysis.count()

print(f'Total POS transactions linked to reservations: {total_pos_with_res:,}')
print(f'Late checkout POS transactions: {late_checkout_count:,} ({late_checkout_count/total_pos_with_res*100:.1f}%)')

# Show late checkout examples
print('\nLate checkout examples:')
late_checkout_analysis.filter(col('is_late_checkout')).select(
    'txn_id', 'res_id', 'check_out_date', 'pos_date', 'timestamp', 'category', 'amount'
).show(5, truncate=False)

## Step 9: Weekend vs Weekday Pricing Analysis (Bonus)

In [0]:
print('\n=== Weekend vs Weekday Pricing Analysis ===')

# Analyze pricing by day of week
pricing_analysis = (
    fact_stays_validated
    .withColumn('day_of_week', dayofweek(col('check_in_date')))
    .withColumn('is_weekend',
        when(col('day_of_week').isin([1, 7]), lit(True))  # 1=Sunday, 7=Saturday
        .otherwise(lit(False)))
    .groupBy('is_weekend')
    .agg(
        count('*').alias('reservation_count'),
        avg('total_price').alias('avg_price'),
        spark_min('total_price').alias('min_price'),
        spark_max('total_price').alias('max_price')
    )
    .orderBy('is_weekend')
)

print('Pricing by weekend vs weekday:')
pricing_analysis.show(truncate=False)

# Calculate weekend premium
pricing_data = pricing_analysis.collect()
if len(pricing_data) == 2:
    weekday_avg = pricing_data[0]['avg_price']
    weekend_avg = pricing_data[1]['avg_price']
    premium = (weekend_avg / weekday_avg - 1) * 100
    print(f'\n📈 Weekend pricing premium: {premium:.1f}%')
    print(f'   Expected: ~100% (2x pricing)')
    if abs(premium - 100) < 10:
        print('   ✅ Dynamic pricing strategy validated!')
    else:
        print('   ⚠️  Weekend premium differs from expected 2x pricing')

## Summary and Documentation

### 🎯 Silver Layer Transformation Complete!

### Data Cleaning Approach

1. **Missing Values**: Filtered out records with missing critical fields (IDs, dates). For non-critical fields, used default values or marked as 'Unknown'.

2. **Date/Timestamp Parsing**: Used `coalesce()` with multiple format patterns and `try_to_timestamp()` to handle various date formats gracefully without throwing errors.

3. **Data Standardization**: Converted all categorical values to proper case (e.g., 'DELUXE' → 'Deluxe') for consistency.

4. **Outlier Handling**: Filtered extreme values (negative prices, prices > $10,000, POS amounts > $5,000).

### Guest Deduplication

- Created `email_base` by removing '+alias' variations from emails
- Used email_base as deduplication key
- Kept the most recent record (by updated_at) for each unique email
- Reduced ~5,150 raw records to ~5,000 unique guests

### SCD Type 2 Implementation

- Used window functions with `lag()` to detect loyalty tier changes
- Created version numbers for each tier change
- Set `valid_from` as the update timestamp and `valid_to` as the next update
- Added `is_current` flag to identify active records
- Created surrogate key (guest_sk) = guest_id + version

### Date Explosion for Room Availability

- Used `sequence()` function to generate array of dates from check_in to check_out (exclusive)
- Applied `explode()` to create one row per night
- Example: 3-night stay creates 3 rows (check-in, +1, +2)

### Overbooking Detection

- Used window function to count bookings per (hotel_id, room_number, date)
- Flagged records where count > 1 as overbooked
- Resolved by keeping first reservation (by res_id) for each date

### Total Folio Calculation

- Aggregated POS transactions by res_id (sum of amounts)
- Left joined with reservations to include those without POS
- total_folio_amount = total_price + total_pos_amount
- Tracked pos_item_count to identify ancillary revenue attachment

### Assumptions Made

1. Check-out date is exclusive (guest leaves that morning)
2. For overbookings, first reservation (lowest res_id) takes precedence
3. POS transactions with NULL res_id are walk-in guests (excluded from unified fact)
4. Late checkout POS (after check_out_date) are valid transactions
5. Default loyalty tier is 'Bronze' for missing values
6. Negative or extreme prices/amounts are data errors and filtered out

---

## Next Steps

You can now proceed to the **Gold Layer transformation** to create KPI tables:
- kpi_revpar (Revenue Per Available Room)
- kpi_adr (Average Daily Rate)
- kpi_ancillary_attachment_rate
- kpi_housekeeping_turnover_time
- kpi_weekend_vs_weekday_revenue

**Shri Radhe Govind Ji! 🙏**